GQA（Grouped-Query Attention，分组查询注意力）是现代大型语言模型（LLM）中极其重要的一项架构优化。要真正理解 GQA 的原理，我们需要弄清楚它是为了解决什么痛点而诞生的。

它的核心痛点就是：**大模型在推理（生成文本）时，显存带宽往往比计算能力更早遇到瓶颈。**



为了解决这个问题，注意力机制经历了一个“演进”的过程。我们可以通过对比三种机制来直观理解 GQA 的原理：

### 1. 传统的基准：MHA (Multi-Head Attention)
在标准的 Transformer 中，我们使用的是多头注意力。
* **原理**：假设模型有 32 个 Query ($Q$) 头，那么它也会严格配备 32 个 Key ($K$) 头和 32 个 Value ($V$) 头。它们是一一对应的关系。
* **痛点 (KV Cache 爆炸)**：在模型逐字生成文本时，为了避免重复计算，我们需要把前面所有生成的 $K$ 和 $V$ 存入显存（即 KV Cache）。MHA 的每一个头都要存一份独立的 KV 向量。当上下文变长或者 Batch Size 变大时，KV Cache 会占用极其恐怖的显存，导致 GPU 显存耗尽，或者因为频繁的内存读写导致推理速度极慢。

### 2. 极端的优化：MQA (Multi-Query Attention)
为了解决 MHA 显存占用过大的问题，研究人员提出了 MQA。
* **原理**：所有的 Query 头全部共享**唯一的一组** Key 头和 Value 头。如果有 32 个 $Q$ 头，它们都只跟这 1 个 $K$ 和 1 个 $V$ 计算注意力。
* **优缺点**：KV Cache 的大小直接缩减到了原来的 1/32，显存占用极低，推理速度飞快。但代价是：模型的表达能力和生成质量出现了明显的下降，因为所有 $Q$ 头被迫关注完全相同的 KV 特征。

### 3. 完美的折中：GQA (Grouped-Query Attention)
GQA 站在了 MHA 和 MQA 的中间，寻找到了“帕累托最优”（Pareto Optimal）。
* **原理**：GQA 将所有的 Query 头进行**分组**（Group）。假设我们有 32 个 $Q$ 头，我们不再像 MQA 那样只给 1 个 KV 头，也不像 MHA 那样给 32 个 KV 头，而是给比如 **8 个** KV 头。
* **运作方式**：这 32 个 $Q$ 头会被分成 8 组，每组包含 4 个 $Q$ 头。同一组内的这 4 个 $Q$ 头，共享它们专属的那 1 个 $K$ 头和 $V$ 头。

---

### GQA 为什么能成功？

1. **大幅降低显存占用**：与 MHA 相比，GQA 的 KV Cache 大小可以成比例地减少（例如只占原来的 1/4 或 1/8），这使得单张显卡能够支持更大的 Batch Size 和更长的上下文。
2. **极高的性价比（维持质量）**：研究表明，不同 $Q$ 头学到的信息是有冗余的。给它们分配适量的独立 KV 头（而不是缩减到 1 个），足以支撑模型捕捉复杂的语义特征。GQA 的生成质量在各项评估中都**极其接近**甚至等同于 MHA。
3. **业界标配**：正因为这种“既要速度、又要质量”的优秀表现，目前主流的开源大模型（如 LLaMA-2、LLaMA-3、Mistral 等）在超过一定参数量时，几乎全部抛弃了原始的 MHA，转而标配 GQA。

很多时候，GQA 的威力只有在结合模型推理阶段的**缓存机制**时才能真正体现。你想了解在逐字生成（Auto-regressive decoding）的过程中，**KV Cache** 具体是如何运作并利用 GQA 节省显存的吗？

In [ ]:
import torch
import torch.nn as nn
import math


class GQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_heads_kv_heads = n_kv_heads
        self.n_group = n_heads // n_kv_heads
        self.head_dim = d_model // n_heads

        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(self.head_dim * n_heads, d_model, bias=False)

    def forward(self, x: torch.Tensor):
        batch_size, seq_len, _ = x.shape

        Q = (
            self.wq(x)
            .view(batch_size, seq_len, self.n_heads, self.head_dim)
            .transpose(1, 2)
        )
        K = (
            self.wk(x)
            .view(batch_size, seq_len, self.n_heads_kv_heads, self.head_dim)
            .transpose(1, 2)
        )

        V = (
            self.wv(x)
            .view(batch_size, seq_len, self.n_heads_kv_heads, self.head_dim)
            .transpose(1, 2)
        )

        # Q :[B NH S D]  K : [B, KVH, S, D]
        # 重复后   Q :[B NH S D]  K : [B, NH, S, D]
        K = torch.repeat_interleave(K, repeats=self.n_group, dim=1)
        V = torch.repeat_interleave(V, repeats=self.n_group, dim=1)

        # scores [B,NH,S,S]
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.head_dim)
        attn_weights = torch.softmax(scores, dim=-1)

        # attn_weight [B,NH,S,S] V K : [B, NH, S, D]
        # output: [B,NH,S,D]
        output = torch.matmul(attn_weights, V)

        output = (
            output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        )

        return self.wo(output)


In [ ]:
def test_modules():
    # 模拟超参数
    batch_size = 2
    seq_len = 10
    d_model = 256
    n_heads = 8         # 8 个 Query 头
    n_kv_heads = 2      # 2 个 KV 头 (意味着每 4 个 Q 头共享 1 个 KV 头)

    print(f"=== 初始化测试参数 ===")
    print(f"Batch Size: {batch_size}, Seq Len: {seq_len}, D_Model: {d_model}")
    print(f"Num Q Heads: {n_heads}, Num KV Heads: {n_kv_heads}\n")

    # 构造随机输入
    # 形状: (batch_size, seq_len, d_model)
    x = torch.randn(batch_size, seq_len, d_model)
    print(f"输入张量形状: {x.shape}")


    # ----------------------------
    # 测试 GQA
    # ----------------------------
    gqa = GQA(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads)
    gqa_out = gqa(x) 
    print(f"GQA 输出张量形状: {gqa_out.shape}")
    assert gqa_out.shape == x.shape, "GQA 输出形状错误！"

    print("\n✅ 所有模块运行成功，前向传播维度完全匹配！")

if __name__ == "__main__":
    test_modules()

=== 初始化测试参数 ===
Batch Size: 2, Seq Len: 10, D_Model: 256
Num Q Heads: 8, Num KV Heads: 2

输入张量形状: torch.Size([2, 10, 256])
GQA 输出张量形状: torch.Size([2, 10, 256])

✅ 所有模块运行成功，前向传播维度完全匹配！
